In [2]:
# --- folders ---
from pathlib import Path  # Import Path: an object-oriented way to work with filesystem paths

BASE = Path("/content/oda_wdi_project")  # Project root folder (Colab writes under /content)

RAW  = BASE / "data" / "raw"             # e.g., /content/oda_wdi_project/data/raw
RAW.mkdir(parents=True, exist_ok=True)   # Create RAW (and any missing parents); don't error if it already exists

PROC = BASE / "data" / "processed"       # e.g., /content/oda_wdi_project/data/processed
PROC.mkdir(parents=True, exist_ok=True)  # Ensure the processed-data folder exists

FIGS = BASE / "reports" / "figures"      # e.g., /content/oda_wdi_project/reports/figures
FIGS.mkdir(parents=True, exist_ok=True)  # Ensure the figures folder exists

REPT = BASE / "reports"                  # e.g., /content/oda_wdi_project/reports
REPT.mkdir(parents=True, exist_ok=True)  # Ensure the reports folder exists

print("Folders ready:", BASE)            # Quick confirmation message


Folders ready: /content/oda_wdi_project


In [4]:
from google.colab import files                  # Colab helper to upload files from your computer

print("Upload these three files:")              # Friendly instructions for the user
print(" - crs_disbursements.csv")
print(" - wdi_indicators.csv")
print(" - pop_income.csv")

uploads = files.upload()                        # Opens a file picker; select all 3 at once
                                                # Returns a dict: {filename: file_bytes}

Upload these three files:
 - crs_disbursements.csv
 - wdi_indicators.csv
 - pop_income.csv


Saving crs_disbursements.csv to crs_disbursements.csv
Saving pop_income.csv to pop_income.csv
Saving wdi_indicators.csv to wdi_indicators.csv


In [5]:
import os  # Standard library module for working with files and paths

# Each assert checks that the file exists at the expected location.
# If any path doesn't exist, Python raises an AssertionError immediately.
assert os.path.exists("/content/crs_disbursements.csv")
assert os.path.exists("/content/wdi_indicators.csv")
assert os.path.exists("/content/pop_income.csv")

print("All good ✅")  # If we got here, all three files are present


All good ✅


In [6]:
# =========================
# ODA + WDI | One-cell build — Fully Commented
# =========================
import pandas as pd, numpy as np, re, os, json, zipfile    # Data wrangling, math, regex, filesystem, JSON, zips
from pathlib import Path                                   # Object-oriented filesystem paths
import matplotlib.pyplot as plt                             # Plotting library

BASE = Path("/content/oda_wdi_project")                    # Project root within Colab's /content
RAW  = BASE/"data"/"raw";        RAW.mkdir(parents=True, exist_ok=True)        # Create raw/ (and parents) if missing
PROC = BASE/"data"/"processed";  PROC.mkdir(parents=True, exist_ok=True)      # Create processed/ if missing
FIGS = BASE/"reports"/"figures"; FIGS.mkdir(parents=True, exist_ok=True)      # Create reports/figures/ if missing
REPT = BASE/"reports";           REPT.mkdir(parents=True, exist_ok=True)       # Create reports/ if missing

crs_fp  = Path("/content/crs_disbursements.csv")          # Input: CRS disbursements
wdi_fp  = Path("/content/wdi_indicators.csv")             # Input: WDI indicators (wide)
meta_fp = Path("/content/pop_income.csv")                 # Input: Population + income group

# Save exact copies of inputs into RAW/ for reproducibility
for fp in [crs_fp, wdi_fp, meta_fp]:
    pd.read_csv(fp, low_memory=False).to_csv(RAW/fp.name, index=False)

# --------- Helpers ---------
def soft_pick(df, options, required=False):
    """Return the first existing column among `options` (case-insensitive).
    If `required` and none found, raise a clear error.
    """
    lo = {c.lower(): c for c in df.columns}                # Map lowercase->original case
    for opt in options:
        if opt.lower() in lo: return lo[opt.lower()]       # Found a match; return original name
    if required: raise ValueError(f"Missing column, tried: {options}")
    return None                                            # Not required and not found

def year_cols(df):
    """Detect columns like '2000' or '2000 [YR2000]' and return their names."""
    out = []
    for c in df.columns:
        if re.fullmatch(r"\d{4}", c): out.append(c)      # Exact 4-digit year
        elif re.fullmatch(r"\d{4}\s*\[YR\d{4}\]", c): out.append(c)  # WDI-style label
    return out

def to_num(x):
    """Convert values to float; tolerate commas and stray strings; return NaN when needed."""
    if pd.isna(x): return np.nan
    if isinstance(x,(int,float,np.number)): return float(x)
    s = str(x).replace(",","" ).strip()
    try: return float(s)
    except: return np.nan

# ---- CRS (Education, Health) ----
crs = pd.read_csv(crs_fp, low_memory=False)                # Load CRS raw data
c_cc   = soft_pick(crs, ["Recipient Code","Recipient3","recipient_code","Country Code","Code"], True)   # Country code
c_cn   = soft_pick(crs, ["Recipient","Recipient name","Recipient Name","Country Name","Country"], True) # Country name
c_year = soft_pick(crs, ["TIME_PERIOD","Time period","Year","year"], True)                               # Year
c_sect = soft_pick(crs, ["Sector4","SECTOR","sector","Purpose (name)","Purpose name"], True)            # Sector
c_val  = soft_pick(crs, ["OBS_VALUE","Observation value","Value","amount","disbursements_constant_usd"], True)  # Amount
# Optional filters to keep constant-USD disbursements only
c_measure = soft_pick(crs, ["MEASURE5","MEASURE","measure"])            # e.g., contains 'disburse'
c_price   = soft_pick(crs, ["Price base","PRICE_BASE"])                   # e.g., contains 'constant'
c_unit    = soft_pick(crs, ["Unit of measure","UNIT_MEASURE"])            # e.g., contains 'US'
c_mult    = soft_pick(crs, ["UNIT_MULT","multiplier"])                    # Power-of-10 multiplier

crs2 = crs.copy()                                           # Work on a copy
if c_measure: crs2["_m"] = crs2[c_measure].astype(str).str.lower()  # Normalize text for filtering
if c_price:   crs2["_p"] = crs2[c_price].astype(str).str.lower()
if c_unit:    crs2["_u"] = crs2[c_unit].astype(str).str.lower()
if c_measure: crs2 = crs2[crs2["_m"].str.contains("disburse", na=False)]  # Keep disbursements
if c_price:   crs2 = crs2[crs2["_p"].str.contains("constant", na=False)]  # Keep constant prices
if c_unit:    crs2 = crs2[crs2["_u"].str.contains("us", na=False)]        # Keep USD

crs2["country_code"] = crs2[c_cc].astype(str).str.upper()   # Standardize country code (upper)
crs2["country_name"] = crs2[c_cn].astype(str).str.strip()    # Trim extra spaces
crs2["year"]         = crs2[c_year].astype(int)             # Parse year as int (assumes clean ints)
crs2["sector"]       = crs2[c_sect].astype(str).str.title()  # Title-case sector
crs2["oda_usd"]      = crs2[c_val].apply(to_num)             # Numeric amount
if c_mult is not None:
    mul = pd.to_numeric(crs2[c_mult], errors="coerce").fillna(0)  # Convert multiplier to numeric
    crs2["oda_usd"] = crs2["oda_usd"] * (10.0 ** mul)            # Apply 10^multiplier
crs2 = crs2[crs2["sector"].isin(["Education","Health"])]        # Keep only target sectors
oda = (
    crs2.groupby(["country_code","country_name","year","sector"], dropna=False)["oda_usd"]
        .sum().reset_index()                                      # Aggregate to country-year-sector
)

# ---- WDI (4 indicators) ----
wdi = pd.read_csv(wdi_fp, low_memory=False)                 # Load WDI wide data
w_cn = soft_pick(wdi, ["Country Name","country_name"], True)        # Country name
w_cc = soft_pick(wdi, ["Country Code","country_code"], True)        # Country code
w_sn = soft_pick(wdi, ["Series Name","indicator_name","Indicator Name"], True)  # Series name
w_sc = soft_pick(wdi, ["Series Code","indicator_code","Indicator Code"], True)  # Series code
w_years = year_cols(wdi); assert w_years, "No WDI year columns."      # Detect year columns

# Wide → long format (one row per country/indicator/year)
wdi_m = wdi.melt(id_vars=[w_cc,w_cn,w_sn,w_sc], value_vars=w_years,
                 var_name="year_raw", value_name="value")
wdi_m["year"]  = wdi_m["year_raw"].str.extract(r"(\d{4})").astype(int)  # Extract numeric year
wdi_m["value"] = pd.to_numeric(wdi_m["value"], errors="coerce")          # Coerce to float

# Keep the four indicators of interest and map to friendly names
keep_codes = {
    "SE.ADT.LITR.ZS": "literacy_rate",
    "SE.SEC.ENRR"   : "school_enrollment",
    "SP.DYN.IMRT.IN": "infant_mortality",
    "SP.DYN.LE00.IN": "life_expectancy",
}
wdi_k = wdi_m[wdi_m[w_sc].isin(keep_codes.keys())].copy()   # Filter 4 codes
wdi_k["indicator"] = wdi_k[w_sc].map(keep_codes)            # Add readable indicator name

# Long → wide (columns for each indicator)
wdi_piv = (wdi_k.pivot_table(index=[w_cc,w_cn,"year"], columns="indicator",
                             values="value", aggfunc="mean")
            .reset_index())
wdi_piv.columns.name = None                                  # Drop pivot index name
wdi_piv.rename(columns={w_cc:"country_code", w_cn:"country_name"}, inplace=True)  # Standardize names

# ---- Population + income group ----
meta = pd.read_csv(meta_fp, low_memory=False)               # Load population + income metadata
m_cc = soft_pick(meta, ["Country Code","country_code","Code"], True)           # Country code
m_cn = soft_pick(meta, ["Country Name","country_name","Country Name_y"], True)  # Country name
m_sc = soft_pick(meta, ["Series Code","indicator_code","Indicator Code"])      # Optional: series code
m_sn = soft_pick(meta, ["Series Name","indicator_name"])                           # Optional: series name
m_inc= soft_pick(meta, ["income_group","Income group","Income Group"])           # Optional: income group

# Helper for years (duplicate of year_cols, scoped here to avoid name clash)
def m_years(df):
    return [c for c in df.columns if re.fullmatch(r"\d{4}", c) or re.fullmatch(r"\d{4}\s*\[YR\d{4}\]", c)]
my = m_years(meta); assert my, "Population metadata must have yearly values."  # Ensure time series present

# Melt meta wide→long; include optional id vars only if present
meta_m = meta.melt(
    id_vars=[m_cc,m_cn,m_sc,m_sn,m_inc] if (m_sc or m_sn) else [m_cc,m_cn,m_inc],
    value_vars=my, var_name="year_raw", value_name="value")
meta_m["year"] = meta_m["year_raw"].str.extract(r"(\d{4})").astype(int)  # Parse year

# Identify the population series rows (by code or by name)
if m_sc is not None:
    pop_rows = meta_m[m_sc].eq("SP.POP.TOTL")
elif m_sn is not None:
    pop_rows = meta_m[m_sn].str.contains("Population, total", case=False, na=False)
else:
    pop_rows = pd.Series(True, index=meta_m.index)            # Assume already filtered

# Build tidy population table with income group
meta_p = meta_m[pop_rows].rename(
    columns={m_cc:"country_code", m_cn:"country_name", "value":"population"})
meta_p["income_group"] = meta_m[m_inc] if m_inc else np.nan
meta_p = meta_p[["country_code","country_name","year","population","income_group"]]
meta_p["population"]   = pd.to_numeric(meta_p["population"], errors="coerce")
meta_p["income_group"] = (meta_p.sort_values(["country_code","year"])              # Income group rarely changes
                          .groupby("country_code")["income_group"].ffill().bfill())  # Fill within country

# ---- Merge + feature engineering ----
if not oda.empty:
    df = (oda.merge(wdi_piv, on=["country_code","country_name","year"], how="left")
             .merge(meta_p, on=["country_code","country_name","year"], how="left"))
else:
    df = wdi_piv.merge(meta_p, on=["country_code","country_name","year"], how="left")
    df["oda_usd"] = np.nan                                   # No CRS data: keep NaNs for ODA

# Literacy lags (1–3 years) for time-aware analyses
for k in (1,2,3):
    if "literacy_rate" in df.columns:
        df[f"Literacy (Lag {k})"] = df.groupby("country_code")["literacy_rate"].shift(k)

# Per-capita and totals
df["disbursements_constant_usd"] = df["oda_usd"]            # Canonical ODA column name
df["ODA per Capita"] = df["disbursements_constant_usd"] / df["population"]  # Guarded by NaNs

# Peer Z-scores within (income_group × year)
def z_by_group(s):
    mu, sd = s.mean(), s.std(ddof=0)
    return (s-mu)/sd if sd and np.isfinite(sd) and sd>0 else np.nan

for src,lab in [("literacy_rate","Z Literacy"),
                ("life_expectancy","Z LifeExp"),
                ("infant_mortality","Z IMR"),
                ("school_enrollment","Z Enrollment")]:
    if src in df.columns:
        df[lab] = df.groupby(["income_group","year"])[src].transform(z_by_group)

# Flag rows with any |Z| >= 2 as notable (potential outliers)
z_cols = [c for c in ["Z Literacy","Z LifeExp","Z IMR","Z Enrollment"] if c in df.columns]
if z_cols:
    df["Mismatch Flag"] = np.where(df[z_cols].abs().max(axis=1) >= 2, "Notable", "Normal")

# Latest snapshot per country (for dashboard cards)
latest = df.sort_values("year").groupby(["country_code","country_name"]).tail(1)
cards = latest[["country_code","year","population","disbursements_constant_usd"]].rename(
    columns={"year":"Latest Year","population":"Latest Population","disbursements_constant_usd":"Latest ODA"})
df = df.merge(cards[["country_code","Latest ODA","Latest Population","Latest Year"]],
              on="country_code", how="left")

# Save main dataset
out_csv = PROC/"final_merged_dataset.csv"
df.to_csv(out_csv, index=False)
print("Saved:", out_csv, "shape:", df.shape)

# Quick figure: total ODA by year (millions USD)
plt.figure(figsize=(9,4))
(df.groupby("year")["disbursements_constant_usd"].sum()/1e6).plot(marker="o")
plt.title("Total ODA (Education + Health) — USD millions")
plt.ylabel("USD (millions)"); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIGS/"oda_total_mln.png", dpi=130); plt.close()

# Summary JSON for quick diagnostics
df_clean = df.dropna(subset=["year"])                       # Only rows with valid years
summary = {
    "rows": int(len(df)),
    "years": [int(df_clean["year"].min()) if not df_clean.empty else None,
              int(df_clean["year"].max()) if not df_clean.empty else None],
    "countries": int(df["country_code"].nunique()),
    "has_cols": [c for c in [
        "literacy_rate","school_enrollment","infant_mortality","life_expectancy",
        "population","income_group","disbursements_constant_usd","ODA per Capita",
        "Literacy (Lag 1)","Literacy (Lag 2)","Literacy (Lag 3)",
        "Z Literacy","Z LifeExp","Z IMR","Z Enrollment",
        "Mismatch Flag","Latest ODA","Latest Population","Latest Year"
    ] if c in df.columns]
}
(REPT/"model_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

# Zip outputs for easy download
zip_path = Path("/content/ODA_WDI_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(out_csv,                   arcname="data/processed/final_merged_dataset.csv")
    z.write(REPT/"model_summary.json", arcname="reports/model_summary.json")
    for fn in os.listdir(FIGS): z.write(FIGS/fn, arcname=f"reports/figures/{fn}")

# In Colab, trigger a download prompt
from google.colab import files
print("ZIP ready:", zip_path)
files.download(str(zip_path))


Saved: /content/oda_wdi_project/data/processed/final_merged_dataset.csv shape: (80, 23)
ZIP ready: /content/ODA_WDI_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>